# WMH-SynthSeg on the Neurodesk A40 server

Step-by-step test of WMH-SynthSeg on the Neurodesk GPU server (NVIDIA A40), from the driver up to
the full script, to show where it fails. Run on the A40 server with the lfbook kernel. Test
scans: sub-03 of the new dataset, one 64mT and one 3T T1w. Outputs go to `work/A40_test/`.

In [10]:
import ctypes
import resource
import sys
import time
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from common import P, NEW_RAW, WORK, THREADS, sh

OUT = WORK / 'A40_test'
OUT.mkdir(parents=True, exist_ok=True)
INF = P['wmh_code'] / 'WMHSynthSeg' / 'inference.py'
SCANS = {'64mT': NEW_RAW / '64mT/sub-03/anat/sub-03_T1w.nii.gz',
         '3T':   NEW_RAW / '3T/sub-03/anat/sub-03_T1w.nii.gz'}
summary = []

## 1. Server

CPUs and memory this container may use, and the GPU as the driver reports it.

In [11]:
mem = Path('/sys/fs/cgroup/memory.max').read_text().strip()
print(f"CPUs: {THREADS} | memory limit: {int(mem) / 2**30:.0f} GB" if mem != 'max' else f'CPUs: {THREADS} | memory: no limit')
sh('nvidia-smi')
summary.append(('1. GPU visible (nvidia-smi)', 'yes'))

CPUs: 24 | memory limit: 89 GB
Tue Oct 13 23:51:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.91.07              Driver Version: 595.91.07      CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A40                     On  |   00000000:00:06.0 Off |                    0 |
|  0%   38C    P8             22W /  300W |       3MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+----------------

## 2. CUDA driver

`cuInit` is the first call of every CUDA program (PyTorch included). 0 means success.

In [12]:
cuda = ctypes.CDLL('libcuda.so.1')
version = ctypes.c_int()
cuda.cuDriverGetVersion(ctypes.byref(version))
err = cuda.cuInit(0)
name = ctypes.c_char_p()
cuda.cuGetErrorName(err, ctypes.byref(name))
print(f'CUDA driver API version: {version.value // 1000}.{version.value % 1000 // 10}')
print(f'cuInit(0) = {err} ({name.value.decode()})')
summary.append(('2. CUDA initialisation (cuInit)', f'failed: {err} {name.value.decode()}' if err else 'ok'))

CUDA driver API version: 13.2
cuInit(0) = 0 (CUDA_SUCCESS)


## 3. PyTorch

In [13]:
import torch

print('torch', torch.__version__, '| built for CUDA', torch.version.cuda)
ok = torch.cuda.is_available()
print('torch.cuda.is_available():', ok)
summary.append(('3. PyTorch sees the GPU', 'yes' if ok else 'no'))

torch 2.14.1+cu130 | built for CUDA 13.0
torch.cuda.is_available(): True


## 4. WMH-SynthSeg code and model

Commit `2bf9a42`, the same as on Colab, with two patches: model path and `weights_only=False`.

In [14]:
sh(f"git -C {P['wmh_code']} log -1 --format='commit %h  %ad'")
sh(f"grep -n 'model_file = \\|torch.load(' {INF}")
print(f"model: {P['wmh_model'].name}, {P['wmh_model'].stat().st_size / 1e6:.0f} MB")
for fs, f in SCANS.items():
    img = nib.load(f)
    print(f"{fs:5s} {f.name}: grid {img.shape[:3]}, voxel {np.round(img.header.get_zooms()[:3], 2)} mm")
summary.append(('4. Code, patches, model', 'ok'))

commit 2bf9a42  Sat Jun 1 15:28:41 2024 -0400
94:    model_file = os.path.join('/home/jovyan/neurodesktop-storage/lfbook/wmh_model', 'WMH-SynthSeg_v10_231110.pth')
127:        checkpoint = torch.load(model_file, map_location=device, weights_only=False)
model: WMH-SynthSeg_v10_231110.pth, 791 MB
64mT  sub-03_T1w.nii.gz: grid (224, 272, 40), voxel [0.81 0.8  5.  ] mm
3T    sub-03_T1w.nii.gz: grid (176, 256, 256), voxel [1. 1. 1.] mm


## 5. WMH-SynthSeg on the GPU

The full script with `--device cuda`, as on Colab, on the 64mT scan.

In [15]:
summary.append(('5. WMH-SynthSeg --device cuda', 'failed (see error below)'))
sh(f"{sys.executable} -u inference.py --i {SCANS['64mT']} --o {OUT / 'gpu_64mT_seg.nii.gz'} --device cuda",
   cwd=INF.parent)

Arguments seem correct; loading Python packages...
/home/jovyan/conda-envs/lfbook/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Using cuda
Using 1 thread(s)
Preparing model and loading weights
Working on image 1 of 1: /home/jovyan/neurodesktop-storage/neurodesk_upload/new/64mT/sub-03/anat/sub-03_T1w.nii.gz
     Loading input volume and normalizing to [0,1]
     Upscaling to target resolution
     Pushing data through the CNN
     Writing segmentation to disk: /home/jovyan/neurodesktop-storage/lfbook/work/A40_test/gpu_64mT_seg.nii.gz
 
All done!
If you use this method in a publication, please cite the following article:
Quantifying white matter hyperintensity and brain volumes in heterogeneous clinical and low-field portable MRI
Laso P, Cerri S, Sorby-Adams A, Guo J, Matteen F, Goebl P, Wu J, Li H, Young SI, Billot B, Puonti O, Rosen MS,
Kirsch J, Strisciuglio 

## 6–7. WMH-SynthSeg on the CPU

The same script with `--device cpu` and all CPUs, first the 64mT scan, then the 3T scan. The 3T
scan takes much longer (about 27 minutes in an earlier run).

In [7]:
def run_cpu(fs):
    out = OUT / f'cpu_{fs}_seg.nii.gz'
    t0 = time.time()
    sh(f"{sys.executable} -u inference.py --i {SCANS[fs]} --o {out} --device cpu --threads {THREADS}",
       cwd=INF.parent)
    mins = (time.time() - t0) / 60
    peak = resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / 2**20   # largest so far, GB
    print(f'\n{fs}: {mins:.1f} min, peak memory {peak:.1f} GB, output {out.name}')
    return mins


summary.append(('6. WMH-SynthSeg --device cpu, 3T', f'ok, {run_cpu("3T"):.1f} min'))

Arguments seem correct; loading Python packages...
/home/jovyan/conda-envs/lfbook/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Using cpu
Using 24 thread(s)
Preparing model and loading weights
Working on image 1 of 1: /home/jovyan/neurodesktop-storage/neurodesk_upload/new/64mT/sub-03/anat/sub-03_T1w.nii.gz
     Loading input volume and normalizing to [0,1]
     Upscaling to target resolution
     Pushing data through the CNN
     Writing segmentation to disk: /home/jovyan/neurodesktop-storage/lfbook/work/A40_test/cpu_64mT_seg.nii.gz
 
All done!
If you use this method in a publication, please cite the following article:
Quantifying white matter hyperintensity and brain volumes in heterogeneous clinical and low-field portable MRI
Laso P, Cerri S, Sorby-Adams A, Guo J, Matteen F, Goebl P, Wu J, Li H, Young SI, Billot B, Puonti O, Rosen MS,
Kirsch J, Strisciuglio 

In [8]:
summary.append(('7. WMH-SynthSeg --device cpu, 3T', f'ok, {run_cpu("3T"):.1f} min'))

Arguments seem correct; loading Python packages...
/home/jovyan/conda-envs/lfbook/lib/python3.11/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
Using cpu
Using 24 thread(s)
Preparing model and loading weights
Working on image 1 of 1: /home/jovyan/neurodesktop-storage/neurodesk_upload/new/3T/sub-03/anat/sub-03_T1w.nii.gz
     Loading input volume and normalizing to [0,1]
     Upscaling to target resolution
     Pushing data through the CNN


KeyboardInterrupt: 

## 8. Summary

In [9]:
pd.DataFrame(summary, columns=['step', 'result'])

,step,result
0,1. GPU visible (nvidia-smi),yes
1,2. CUDA initialisation (cuInit),ok
2,3. PyTorch sees the GPU,yes
3,"4. Code, patches, model",ok
4,5. WMH-SynthSeg --device cuda,failed (see error below)
5,"6. WMH-SynthSeg --device cpu, 64mT","ok, 1.0 min"
